In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader



# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
     transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])



# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

import matplotlib.pyplot as plt
import numpy as np
# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [50,40,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)
    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
model = efficientnet_v2_s(pretrained = True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval().to(device)

# Freeze all parameters in the feature extractor
for param in model.features.parameters():
            param.requires_grad = False

# print(model)


In [ ]:
# Write your code here

import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from torch import nn



# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    running_loss = 0.0


    for images, labels in tqdm(dataloader):
        # print(f'Masks size: {masks.shape}'); break # CEntropyL expects size (BZ, H, W) so squeeze ([32, 1, 256, 256]) -- > [32, 256, 256]
        images, labels = images.to(device), labels.to(device)
        # images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]
        optimizer.zero_grad()
        labels-= 1
        # images = SubtractOne(images)
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Track accuracy
        correct_predictions += (torch.softmax(outputs, dim=1).argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return running_loss / len(dataloader), correct_predictions / total_samples

# 🔹 Validation Loop
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Remember to use torch.no_grad() context manager
    with torch.no_grad():

      for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        labels-= 1

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        # Track accuracy
        correct_predictions += (torch.softmax(outputs, dim=1).argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    return running_loss / len(dataloader), correct_predictions / total_samples


In [ ]:
# Write your code here
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.classifier.parameters(), lr=0.001)  # YOUR CODE HERE


num_epochs = 5
# Lists to store metrics

train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print("Starting Training...")


# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_accuracy = validate_epoch(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={test_loss:.4f}, Val Accuracy={test_accuracy:.2f}%")

In [ ]:
# Plot loss curve
plt.figure(figsize=(12, 5))
#          1 row 2 columns means 2 subplots
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), test_losses, label="Val Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()

# Subplot 2: Plot training and validation accuracy

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), test_accuracies, label="Val Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title('Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here

# 🔹 Validation Loop
def TTAvalidate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():

      for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        h_flipped = torch.flip(images, dims=[3])
        v_flipped = torch.flip(images, dims=[2])
        avg = (outputs + h_flipped + v_flipped) / 3
        loss = criterion(avg, labels)
        running_loss += loss.item()

        # Track accuracy
        correct_predictions += (torch.softmax(outputs, dim=1).argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    return running_loss / len(dataloader), correct_predictions / total_samples


In [ ]:
test_losses = []
test_accuracies = []

print("Starting Training...")


# Training process
for epoch in range(num_epochs):
    test_loss, test_accuracy = TTAvalidate_epoch(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Val Loss={test_loss:.4f}, Val Accuracy={test_accuracy:.2f}%")